# 01 – SFT: den Denkprozess lernen
Das Modell lernt die **Methode**: Annahmen aufdecken → Denkwerkzeuge anwenden → Rohideen → ehrlich aussortieren → beste Idee ausarbeiten.
Trainiert wird bewusst **ohne Business-Beispiele**; Geschäftsideen kommen nur im Test vor.

**Kaggle:** GPU T4 x2 · Internet an · Dauer ca. 1,5–2,5 h → am besten über **Save Version → Save & Run All (Commit)** im Hintergrund laufen lassen.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if os.path.exists("repo"):
    subprocess.run(["git", "-C", "repo", "pull", "-q"], check=True)   # immer neuester Stand
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json, re, torch

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: eingebauten Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

def generate(model, tokenizer, question, max_new_tokens=600, temperature=0.8):
    p = chat(tokenizer, [{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def response_parts(model_name):
    if "qwen" in model_name.lower():
        return "<|im_start|>user\n", "<|im_start|>assistant\n"
    return "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"], max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
L = CFG["lora"]
model = FastLanguageModel.get_peft_model(
    model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
from datasets import load_dataset
to_text = lambda r: {"text": chat(tokenizer, r["messages"])}
train = load_dataset("json", data_files="repo/data/sft.jsonl", split="train").map(to_text)
val   = load_dataset("json", data_files="repo/data/sft_val.jsonl", split="train").map(to_text)
print(f"Training: {len(train)} · Prüfset: {len(val)}")
print(train[0]["text"][:900])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
S = CFG["sft"]
steps_per_epoch = max(1, len(train) // (S["batch_size"] * S["grad_accum"]))
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=train, eval_dataset=val,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=CFG["max_seq_length"],
        per_device_train_batch_size=S["batch_size"], gradient_accumulation_steps=S["grad_accum"],
        per_device_eval_batch_size=1,
        num_train_epochs=S["epochs"], learning_rate=S["lr"],
        warmup_steps=max(1, int(0.05 * steps_per_epoch * S["epochs"])),
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps", eval_steps=S["eval_steps"],
        save_strategy="steps", save_steps=S["eval_steps"], save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        output_dir="checkpoints_sft", report_to="none", seed=42))
trainer = train_on_responses_only(trainer, *response_parts(CFG["base_model"]))
trainer.train()
# Lesen: "Validation Loss" sollte sinken. Steigt er wieder, während "Training Loss" weiter fällt,
# lernt das Modell auswendig – das beste Checkpoint davor wird automatisch behalten.

In [ ]:
# Verlauf Training vs. Prüfset
import pandas as pd
log = pd.DataFrame(trainer.state.log_history)
print(log[["step","loss"]].dropna().tail(10).to_string(index=False))
print(log[["step","eval_loss"]].dropna().to_string(index=False))
print("Bestes Checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
# Transfer-Test: Business kam im Training NIE vor. Basis vs. trainiert.
FastLanguageModel.for_inference(model)
tests = [json.loads(l)["prompt"] for l in open("repo/eval/business_holdout.jsonl")][:2]
for q in tests:
    print("="*90, "\nFRAGE:", q)
    with model.disable_adapter():
        print("\n--- BASISMODELL ---\n", generate(model, tokenizer, q)[:1500])
    print("\n--- TRAINIERT ---\n", generate(model, tokenizer, q)[:1500])

In [ ]:
model.save_pretrained("adapter_sft"); tokenizer.save_pretrained("adapter_sft")
print("Gespeichert: /kaggle/working/adapter_sft")